******Car-Dheko_Used_Car_Price_Prediction******

There are 6 files of cities data

****Data Processing****

**a)	Import and concatenate:**

i)	Import all city’s dataset which is in unstructured format.

ii)	Convert it into a  structured format.

iii)Added a new column named ‘City’ and assign values for all rows with the name of the respective city.

iv)	Concatenate all datasets and make it as a single dataset.

**Banglore**

Perfectly Done unstructured to structured Banglore cars

In [10]:
import pandas as pd 
import ast
import os

def load_and_parse_data(file_path):
    """Load and parse the CSV file with nested JSON data"""
    try:
        df = pd.read_csv(file_path)
        
        def parse_dict_column(column):
            return column.apply(lambda x: ast.literal_eval(x) if isinstance(x, str) and x.strip().startswith('{') else x)
        
        for col in ["new_car_detail", "new_car_overview", "new_car_feature", "new_car_specs"]:
            if col in df.columns:
                df[col] = parse_dict_column(df[col])
        
        return df
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None

def flatten_data(df):
    """Flatten all nested structures into a single DataFrame"""
    try:
        # OVERVIEW
        def flatten_overview(overview_data):
            if isinstance(overview_data, dict):
                return {item['key']: item['value'] for item in overview_data.get("top", []) if isinstance(item, dict)}
            return {}
        
        df_overview = df["new_car_overview"].apply(flatten_overview) if "new_car_overview" in df.columns else pd.DataFrame()

        # DETAIL
        df_detail = pd.json_normalize(df["new_car_detail"]) if "new_car_detail" in df.columns else pd.DataFrame()

        # FEATURE
        def flatten_feature(data):
            features = []
            if isinstance(data, dict):
                top = data.get("top", [])
                features.extend(f.get("value") for f in top if isinstance(f, dict))
                for section in data.get("data", []):
                    if isinstance(section, dict):
                        for item in section.get("list", []):
                            if isinstance(item, dict):
                                val = item.get("value")
                                if val:
                                    features.append(val)
            return {'Feature_' + str(i): v for i, v in enumerate(features)}
        
        df_feature = df["new_car_feature"].apply(flatten_feature) if "new_car_feature" in df.columns else pd.DataFrame()

        # SPECS
        def flatten_specs(data):
            specs = {}
            if isinstance(data, dict):
                for item in data.get("top", []):
                    if isinstance(item, dict):
                        k = item.get("key")
                        v = item.get("value")
                        if k and v:
                            specs[k] = v
                for section in data.get("data", []):
                    if isinstance(section, dict):
                        for item in section.get("list", []):
                            if isinstance(item, dict):
                                k = item.get("key")
                                v = item.get("value")
                                if k and v:
                                    specs[k] = v
            return specs
        
        df_specs = df["new_car_specs"].apply(flatten_specs) if "new_car_specs" in df.columns else pd.DataFrame()

        # Convert all dict results to DataFrames
        df_overview = pd.DataFrame(df_overview.tolist())
        df_feature = pd.DataFrame(df_feature.tolist())
        df_specs = pd.DataFrame(df_specs.tolist())

        # Combine all DataFrames
        base_cols = df[["car_links"]] if "car_links" in df.columns else pd.DataFrame()
        df_final = pd.concat([base_cols, df_detail, df_overview, df_feature, df_specs], axis=1)
        df_final["City"] = "bangalore"
        
        return df_final
    except Exception as e:
        print(f"❌ Error flattening data: {e}")
        return None

def save_cleaned_data(df, output_dir="output"):
    """Save the cleaned DataFrame to csv"""
    try:
        os.makedirs(output_dir, exist_ok=True)
        output_path = os.path.join(output_dir, "structured_bangalore_cars.csv")
        df.to_csv(output_path, index=False)
        print(f"✅ Saved cleaned data to: {output_path}")
        return True
    except Exception as e:
        print(f"❌ Error saving data: {e}")
        return False

def main():
    input_path = "csv_files/bangalore_cars.csv"
    df = load_and_parse_data(input_path)
    if df is None:
        return

    df_flat = flatten_data(df)
    if df_flat is None:
        return

    save_cleaned_data(df_flat)

if __name__ == "__main__":
    main()


✅ Saved cleaned data to: output\structured_bangalore_cars.csv


In [14]:
df_final["City"] 

0       bangalore
1       bangalore
2       bangalore
3       bangalore
4       bangalore
          ...    
1476    bangalore
1477    bangalore
1478    bangalore
1479    bangalore
1480    bangalore
Name: City, Length: 1481, dtype: object

             Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0    new_car_detail    object            1481          0             0.0
1  new_car_overview    object            1481          0             0.0
2   new_car_feature    object            1481          0             0.0
3     new_car_specs    object            1481          0             0.0
4         car_links    object            1481          0             0.0


In [15]:
nan_summary = pd.DataFrame({
    'Column': df_final .columns,
    'Data Type': df_final .dtypes,
    'Non-Null Count': df_final.notna().sum(),
    'NaN Count': df_final .isna().sum(),
    'NaN Percentage': (df_final.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                       Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                          it     int64            1481          0            0.00
1                          ft    object            1481          0            0.00
2                          bt    object            1481          0            0.00
3                          km    object            1481          0            0.00
4                transmission    object            1481          0            0.00
5                     ownerNo     int64            1481          0            0.00
6                       owner    object            1481          0            0.00
7                         oem    object            1481          0            0.00
8                       model    object            1481          0            0.00
9                   modelYear     int64            1481          0            0.00
10           centralVariantId     int64            1481          0            0.00
11  

In [34]:
df_final[['price','bt', 'Kms Driven','owner' ,'Color','Year of Manufacture','Mileage', 'RTO','Fuel Type','Registration Year','modelYear', 'Insurance Validity','Gear Box', 'modelYear', 'Transmission', 'Seats', 'City', 'Engine Displacement']].head(10)

,price,bt,Kms Driven,owner,Color,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,modelYear,Transmission,Seats,Seats,City,Engine Displacement
0,₹ 4 Lakh,Hatchback,"1,20,000 Kms",3rd Owner,White,2015.0,23.1 kmpl,KA51,Petrol,2015,2015,Third Party insurance,5 Speed,2015,Manual,5 Seats,5,bangalore,998 cc
1,₹ 8.11 Lakh,SUV,"32,706 Kms",2nd Owner,White,2018.0,17 kmpl,KA05,Petrol,Feb 2018,2018,Comprehensive,5 Speed,2018,Manual,5 Seats,5,bangalore,1497 cc
2,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,Red,2018.0,23.84 kmpl,KA03,Petrol,Sept 2018,2018,Comprehensive,5 Speed,2018,Manual,5 Seats,5,bangalore,1199 cc
3,₹ 4.62 Lakh,Sedan,"17,794 Kms",1st Owner,Others,2014.0,19.1 kmpl,KA53,Petrol,Dec 2014,2014,Comprehensive,5 Speed,2014,Manual,5 Seats,5,bangalore,1197 cc
4,₹ 7.90 Lakh,SUV,"60,000 Kms",1st Owner,Gray,2015.0,23.65 kmpl,KA04,Diesel,2015,2015,Third Party insurance,5 Speed,2015,Manual,5 Seats,5,bangalore,1248 cc
5,₹ 19 Lakh,SUV,"20,000 Kms",1st Owner,Others,2020.0,17.1 kmpl,KA04,Diesel,2020,2020,Third Party insurance,6 Speed,2020,Manual,5 Seats,5,bangalore,1956 cc
6,₹ 3.45 Lakh,Hatchback,"37,772 Kms",1st Owner,Grey,2017.0,20.63 kmpl,KA05,Petrol,Aug 2017,2017,Comprehensive,5 Speed,2017,Manual,5 Seats,5,bangalore,1198 cc
7,₹ 12 Lakh,SUV,"30,000 Kms",1st Owner,Others,2021.0,18.15 kmpl,KA51,Petrol,2021,2021,Third Party insurance,7-Speed,2021,Automatic,5 Seats,5,bangalore,998 cc
8,₹ 9.60 Lakh,Sedan,"37,000 Kms",1st Owner,Maroon,2018.0,20.28 kmpl,KA03,Petrol,Aug 2018,2018,Comprehensive,4 Speed,2018,Automatic,5 Seats,5,bangalore,1462 cc
9,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,Red,2017.0,23.84 kmpl,KA03,Petrol,Jan 2018,2017,Comprehensive,5 Speed,2017,Manual,5 Seats,5,bangalore,1199 cc


✅ Selected Features and Justifications

| Column               | Description                                   | Justification                                                                 |
|----------------------|-----------------------------------------------|-------------------------------------------------------------------------------|
| `price`              | Selling price of the used car (Target variable) | This is the variable we're predicting.                                        |
| `bt`                 | Possibly body type or build type              | Vehicle type (e.g., SUV, sedan) affects demand, pricing, and buyer preference.|
| `Kms Driven`         | Total kilometers driven                        | Indicates vehicle usage; higher values usually reduce resale value.           |
| `owner`              | Number or type of previous owners              | Helps assess usage history and trust; fewer owners often mean better value.   |
| `Color`              | Exterior color of the car                      | Certain colors have higher resale appeal depending on regional trends.        |
| `Year of Manufacture`| Production year of the car                     | Indicates age; newer cars typically sell for higher prices.                   |
| `Mileage`            | Fuel efficiency (e.g., km/l)                   | Higher mileage is appealing and adds value to the car.                        |
| `RTO`                | Regional Transport Office location             | RTO location can influence resale value due to local tax rates and rules.     |
| `Fuel Type`          | Petrol, Diesel, CNG, Electric, etc.            | Different fuels affect running costs and buyer demand.                        |
| `Registration Year`  | Year the car was registered                    | Might differ from manufacturing year; important for insurance and resale.     |
| `modelYear`          | Year of manufacture                            | Reflects the age of the vehicle; newer cars tend to sell for higher prices.   |
| `Insurance Validity` | Remaining insurance period                     | A valid insurance policy adds value and trust for the buyer.                  |
| `Gear Box`           | Number of gears                                | Indicates car performance and class; more gears can mean better performance.  |
| `Transmission`       | Manual or Automatic                            | Automatics generally command a higher resale price, especially in cities.     |
| `Seats`              | Number of seats                                | More seating capacity appeals to families and commercial buyers.              |
| `City`               | City where the car is listed                   | Price trends vary by location due to demand, taxes, and road conditions.      |
| `Engine Displacement`| Engine size in CC                              | Affects performance, tax class, and buyer interest.                           |


Banglore necessary  columns

In [15]:
import pandas as pd
import os

# Load the Excel file
file_path = "output/structured_bangalore_cars.csv"
df = pd.read_csv(file_path)

# Define wanted columns
wanted_columns = [

    'price',
    'bt',
    'Kms Driven',
    'owner' ,
    
    'Year of Manufacture',
    'Mileage',
    'RTO',
    'Fuel Type',
    'Registration Year',
    'modelYear',
    'Insurance Validity',
    'Gear Box',
    'Transmission',
    'Seats',
    'City',
    'Engine Displacement'
  
]

# Filter only the columns that exist in the DataFrame
wanted_columns_present = [col for col in wanted_columns if col in df.columns]

# Select and save the cleaned data
df_cleaned = df[wanted_columns_present]

# Make sure the output folder exists
os.makedirs("output", exist_ok=True)

# Save the DataFrame to Excel inside the output folder
df_cleaned.to_csv("output/banglore_wanted.csv", index=False)

print("✅ File saved to: output/banglore_wanted.csv")


✅ File saved to: output/banglore_wanted.csv


In [16]:
nan_summary = pd.DataFrame({
    'Column': df_cleaned .columns,
    'Data Type': df_cleaned .dtypes,
    'Non-Null Count': df_cleaned.notna().sum(),
    'NaN Count': df_cleaned .isna().sum(),
    'NaN Percentage': (df_cleaned.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price    object            1481          0            0.00
1                    bt    object            1481          0            0.00
2            Kms Driven    object            1481          0            0.00
3                 owner    object            1481          0            0.00
4   Year of Manufacture   float64            1474          7            0.47
5               Mileage    object            1439         42            2.84
6                   RTO    object            1313        168           11.34
7             Fuel Type    object            1481          0            0.00
8     Registration Year    object            1474          7            0.47
9             modelYear     int64            1481          0            0.00
10   Insurance Validity    object            1478          3            0.20
11             Gear Box    object            1457         24            1.62

***b)Handling Missing Values:***

 Identify and fill or remove missing values in the dataset. 

i)	For numerical columns, use techniques like mean, median, or mode imputation.

ii)	For categorical columns, use mode imputation or create a new category for missing values.


In [4]:
import pandas as pd
import numpy as np
import os

# Load the dataset
df = pd.read_csv("output/banglore_wanted.csv")

# Function to print missing value statistics
def print_missing_stats(df, title="Before Imputation"):
    print(f"\n🔍 {title} Missing Values Summary:")
    print("="*60)
    missing_data = df.isnull().sum()
    total_rows = len(df)
    missing_percent = (missing_data / total_rows) * 100
    
    stats_df = pd.DataFrame({
        'Missing Values': missing_data,
        '% Missing': missing_percent.round(2)
    })
    
    print(stats_df[stats_df['Missing Values'] > 0].sort_values('% Missing', ascending=False))
    print("="*60)
    print(f"Total rows in dataset: {total_rows}\n")

# Initial missing value analysis
print_missing_stats(df)

# Create a copy for tracking changes
df_cleaned_filled = df.copy()

# Dictionary to store imputation details
imputation_report = {}

# Handle numerical columns
numerical_cols = ['Year of Manufacture']
for col in numerical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        median_val = df_cleaned_filled[col].median()
        df_cleaned_filled[col].fillna(median_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()
        
        imputation_report[col] = {
            'type': 'numerical',
            'before': before,
            'filled': before - after,
            'method': 'median',
            'value': median_val,
            'justification': 'Median is robust against outliers in manufacturing years'
        }

# Handle categorical columns
categorical_cols = ['Color', 'RTO', 'Registration Year', 'Insurance Validity', 
                   'Gear Box', 'Seats', 'Engine Displacement', 'Mileage']

for col in categorical_cols:
    if col in df_cleaned_filled.columns:
        before = df_cleaned_filled[col].isnull().sum()
        mode_val = df_cleaned_filled[col].mode()[0]
        df_cleaned_filled[col].fillna(mode_val, inplace=True)
        after = df_cleaned_filled[col].isnull().sum()
        
        imputation_report[col] = {
            'type': 'categorical',
            'before': before,
            'filled': before - after,
            'method': 'mode',
            'value': mode_val,
            'justification': 'Most frequent value is appropriate for categorical data'
        }

# Special handling for 'Mileage' column
if 'Mileage' in df_cleaned_filled.columns:
    # Extract numerical part from Mileage (e.g., '23.1 kmpl' → 23.1)
    df_cleaned_filled['Mileage'] = df_cleaned_filled['Mileage'].str.extract('(\d+\.?\d*)').astype(float)
    
    before = df_cleaned_filled['Mileage'].isnull().sum()
    median_mileage = df_cleaned_filled['Mileage'].median()
    df_cleaned_filled['Mileage'].fillna(median_mileage, inplace=True)
    after = df_cleaned_filled['Mileage'].isnull().sum()
    
    imputation_report['Mileage'] = {
        'type': 'converted numerical',
        'before': before,
        'filled': before - after,
        'method': 'median',
        'value': median_mileage,
        'justification': 'After converting string to numerical, median is robust for mileage values'
    }

# Final missing value analysis
print_missing_stats(df_cleaned_filled, "After Imputation")

# Generate imputation report
print("\n✅ Imputation Summary")
print("="*60)
print("Here's a breakdown of how missing values were handled in your dataset:")
print("-"*60)

report_df = pd.DataFrame.from_dict(imputation_report, orient='index')
report_df = report_df[['type', 'before', 'filled', 'method', 'value', 'justification']]
report_df.columns = ['Type', 'Missing Before', 'Filled', 'Strategy', 'Value Used', 'Justification']

print(report_df.sort_values('Missing Before', ascending=False))
print("="*60)

# Detect and print data types
print("\n🔎 Final Data Types:")
print("="*60)
print(df_cleaned_filled.dtypes)
print("="*60)

# df_cleaned_filled = df_cleaned_filled.convert_dtypes()  # Automatically converts object → string, numbers → Int64/Float64
# print(df_cleaned_filled.dtypes)
# print("automatically_detected the datatypes")

# Create a copy to detect data types without modifying the original
df_detected_dtypes = df_cleaned_filled.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_cleaned_filled.dtypes)
print("\nDetected dtypes (without saving):")
print(df_detected_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")

# Save the cleaned data
os.makedirs("output", exist_ok=True)
output_path = "output/banglore_filled_removed.csv"
df_detected_dtypes.to_csv(output_path, index=False)

print(f"\n✅ Cleaned dataset saved to: {output_path}")


🔍 Before Imputation Missing Values Summary:
                     Missing Values  % Missing
RTO                             168      11.34
Mileage                          42       2.84
Gear Box                         24       1.62
Year of Manufacture               7       0.47
Registration Year                 7       0.47
Insurance Validity                3       0.20
Engine Displacement               3       0.20
Seats                             1       0.07
Total rows in dataset: 1481


🔍 After Imputation Missing Values Summary:
Empty DataFrame
Columns: [Missing Values, % Missing]
Index: []
Total rows in dataset: 1481


✅ Imputation Summary
Here's a breakdown of how missing values were handled in your dataset:
------------------------------------------------------------
                                    Type  Missing Before  Filled Strategy  \
RTO                          categorical             168     168     mode   
Gear Box                     categorical              24   

C:\Users\USER\AppData\Local\Temp\ipykernel_6524\255457326.py:40: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_cleaned_filled[col].fillna(median_val, inplace=True)
C:\Users\USER\AppData\Local\Temp\ipykernel_6524\255457326.py:60: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For 

In [5]:
nan_summary = pd.DataFrame({
    'Column': df_detected_dtypes .columns,
    'Data Type': df_detected_dtypes .dtypes,
    'Non-Null Count': df_detected_dtypes.notna().sum(),
    'NaN Count': df_detected_dtypes .isna().sum(),
    'NaN Percentage': (df_detected_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price  string[python]            1481          0             0.0
1                    bt  string[python]            1481          0             0.0
2            Kms Driven  string[python]            1481          0             0.0
3                 owner  string[python]            1481          0             0.0
4   Year of Manufacture           Int64            1481          0             0.0
5               Mileage         Float64            1481          0             0.0
6                   RTO  string[python]            1481          0             0.0
7             Fuel Type  string[python]            1481          0             0.0
8     Registration Year  string[python]            1481          0             0.0
9             modelYear           Int64            1481          0             0.0
10   Insurance Validity  string[python]            1481          0             0.0
11  

***c)	Standardising Data Formats:***

i)	Check for all data types and do the necessary steps to keep the data in the correct format.

(1)	Eg. If a data point has string formats like 70 kms, then remove the unit ‘kms’ and change the data type from string to integers.


In [6]:
df_detected_dtypes.head()

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,₹ 4 Lakh,Hatchback,"1,20,000 Kms",3rd Owner,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,998 cc
1,₹ 8.11 Lakh,SUV,"32,706 Kms",2nd Owner,2018,17.0,KA05,Petrol,Feb 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1497 cc
2,₹ 5.85 Lakh,Hatchback,"11,949 Kms",1st Owner,2018,23.84,KA03,Petrol,Sept 2018,2018,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1199 cc
3,₹ 4.62 Lakh,Sedan,"17,794 Kms",1st Owner,2014,19.1,KA53,Petrol,Dec 2014,2014,Comprehensive,5 Speed,Manual,5 Seats,bangalore,1197 cc
4,₹ 7.90 Lakh,SUV,"60,000 Kms",1st Owner,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5 Speed,Manual,5 Seats,bangalore,1248 cc


| Transmission Type | Gear Number        | Justification                                                                                                                                                              |
|-------------------|--------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| CVT               | Theoretically Infinite | Uses a belt and pulley (or cone) system that continuously adjusts the ratio, offering a seamless and theoretically infinite number of gear ratios within its operating range. |
| Direct Drive      | Typically One (1:1) | Represents a single, direct connection where the input and output shafts rotate at the same speed, with no gear reduction or multiplication.                               |
| IVT               | Theoretically Infinite | Hyundai/Kia's marketing term for their CVT technology, functioning on the same principle of continuous ratio adjustment via a belt and pulley system.                   |
| iMT               | Fixed (e.g., 5 or 6) | A manual transmission with a traditional set of physical gears. The electronic system automates the clutch, but the number of selectable gears remains fixed.             |
| AGS               | Fixed (e.g., 5)    | An automated manual transmission that uses a traditional set of physical gears. The system automates both clutch operation and gear selection, but the number of gears is fixed. |

In [10]:
import pandas as pd
from decimal import Decimal
import os
import re


try:
    df_standardised = pd.read_csv("output/banglore_filled_removed.csv")  # Ensure correct file name
except FileNotFoundError:
    print("Error: The file 'output/banglore_filled_removed.csv' was not found.")
    exit()


def clean_numeric_column(series, pattern=None, dtype=float):
    """Helper function to clean numeric columns"""
    try:
        if series.dtype == object:
            if pattern:
                extracted = series.str.extract(pattern)[0]
            else:
                extracted = series.astype(str).str.replace(r'[^\d\.]', '', regex=True) # Keep decimals
            if dtype == 'Int64':
                return pd.to_numeric(extracted, errors='coerce').astype('Int64')
            return pd.to_numeric(extracted, errors='coerce')
        return series
    except Exception as e:
        print(f"⚠️ Warning cleaning column {series.name}: {e}")
        return series
    
def clean_numeric_series(series, dtype=float, pattern=None):
    """Clean a pandas Series containing numeric values"""
    try:
        if not pd.api.types.is_string_dtype(series):
            series = series.astype(str)
            
        # Remove commas and extract numeric values
        cleaned = series.str.replace(',', '')
        
        if pattern:
            extracted = cleaned.str.extract(pattern)[0]
        else:
            if dtype == float:
                extracted = cleaned.str.extract(r'([\d\.]+)')[0]
            else:
                extracted = cleaned.str.extract(r'(\d+)')[0]
        
        # Convert to appropriate type
        if dtype == 'Int64':
            return pd.to_numeric(extracted, errors='coerce').astype('Int64')
        return pd.to_numeric(extracted, errors='coerce').astype(dtype)
    except Exception as e:
        print(f"⚠️ Warning cleaning numeric series: {e}")
        return series
    
def clean_kms_driven(kms_str):
    """
    Cleans the 'Kms Driven' string by removing non-numeric characters
    and converting it to an integer.
    Handles cases with commas and the "Kms" suffix.
    Returns the cleaned number as an integer or None if cleaning fails.
    """
    if isinstance(kms_str, (int, float)):
        return int(kms_str)  # Already numeric

    cleaned_str = re.sub(r'[^\d]', '', str(kms_str))  # Remove non-digits
    if cleaned_str:
        return int(cleaned_str)
    return None

if 'Kms Driven' in df_standardised.columns:
    df_standardised['Kms Driven'] = df_standardised['Kms Driven'].apply(clean_kms_driven)
    print("✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer")

def convert_price_to_numeric(price_str):
    """
    Converts price strings in '₹ X.XX Lakh', '₹ X Lakh', or '₹ X.XX Crore' format to numeric INR.
    Handles potential non-numeric values by returning NaN.
    """
    if isinstance(price_str, (int, float, Decimal)):
        return float(price_str)  # Already numeric

    price_str = str(price_str).strip().replace(',', '')  # Remove leading/trailing whitespace and commas

    lakh_match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Lakh', price_str, re.IGNORECASE)
    if lakh_match:
        return float(Decimal(lakh_match.group(1)) * Decimal('100000'))

    crore_match = re.search(r'₹\s*(\d+(\.\d+)?)\s*Crore', price_str, re.IGNORECASE)
    if crore_match:
        return float(Decimal(crore_match.group(1)) * Decimal('10000000'))

    # Handle cases with just numeric values (after removing '₹' and spaces)
    numeric_only = price_str.replace('₹', '').strip()
    if numeric_only.isdigit() or ('.' in numeric_only and all(c.isdigit() or c == '.' for c in numeric_only)):
        try:
            return float(numeric_only)
        except ValueError:
            return pd.NA  # Return NaN for invalid numeric strings

    return pd.NA  # Return NaN for non-convertible strings

if 'price' in df_standardised.columns:
    original_prices = df_standardised['price'].copy()
    df_standardised['price_numeric'] = df_standardised['price'].apply(convert_price_to_numeric)

    # Count NaN values after conversion
    nan_count = df_standardised['price_numeric'].isna().sum()

    # Identify non-convertible original values
    non_convertible_original = original_prices[df_standardised['price_numeric'].isna()].unique().tolist()

    # Update the original 'price' column with the numeric conversions
    df_standardised['price'] = df_standardised['price_numeric']

    print("✅ Processed 'price' column - attempted to convert 'Lakh' and 'Crore' values to numeric INR")
    print(f"Number of NaN values in 'price' column after conversion: {nan_count}")
    if non_convertible_original:
        print(f"\n⚠️ The following original values in the 'price' column could not be converted to numeric: {non_convertible_original}")
    else:
        print("\n🎉 All identified price values were successfully converted to numeric.")

    # Optionally, you can drop the temporary 'price_numeric' column
    if 'price_numeric' in df_standardised.columns:
        df_standardised.drop(columns=['price_numeric'], inplace=True)



def extract_gear_speeds_v4_modified(gear_box_str):
    """
    Extracts the number of speeds from a Gear Box string.
    Handles various formats including explicit speed numbers and common transmission types,
    assigning specific numerical values to CVT, Direct Drive, IVT, iMT, and AGS.
    Returns an integer representing the number of speeds or a specific code for other types.
    """
    if isinstance(gear_box_str, (int, float)):
        return int(gear_box_str)

    gear_box_str = str(gear_box_str).strip().lower()

    # Explicit number of speeds
    speed_match = re.search(r'(\d+)\s*speed', gear_box_str)
    if speed_match:
        return int(speed_match.group(1))

    speed_match_hyphen = re.search(r'(\d+)-speed', gear_box_str)
    if speed_match_hyphen:
        return int(speed_match_hyphen.group(1))

    # Leading number (e.g., 8G-DCT)
    leading_number_match = re.search(r'^(\d+)', gear_box_str)
    if leading_number_match:
        return int(leading_number_match.group(1))

    # Handle specific transmission types with assigned numerical values
    if 'cvt' in gear_box_str or 'ivt' in gear_box_str:
        return 0  # Or -1 to represent continuous
    elif 'direct drive' in gear_box_str:
        return 1
    elif 'imt' in gear_box_str:
        return 5  # Or 6, depending on the common range
    elif 'ags' in gear_box_str:
        return 5  # Or 6, depending on the common range

    # Handle manual transmissions (explicitly mention number of speeds)
    manual_match = re.search(r'(\d+)\s*speed\s*manual(?: transmission)?', gear_box_str)
    if manual_match:
        return int(manual_match.group(1))
    manual_match_short = re.search(r'five speed manual', gear_box_str) # Specific case
    if manual_match_short:
        return 5

    return pd.NA  # Return NaN for truly unidentifiable cases

if 'Gear Box' in df_standardised.columns:
    original_gear_box = df_standardised['Gear Box'].copy()
    df_standardised['Gear Box_numeric'] = df_standardised['Gear Box'].apply(extract_gear_speeds_v4_modified)

    # Update the original 'Gear Box' column with the numeric conversions
    df_standardised['Gear Box'] = df_standardised['Gear Box_numeric']

    print("✅ Processed 'Gear Box' column - attempted to extract the number of speeds (version 4 - modified for ML)")
    print(f"Number of NaN values in 'Gear Box' column after extraction: {df_standardised['Gear Box'].isnull().sum()}")
    if df_standardised['Gear Box'].isnull().sum() > 0:
        print(f"\n⚠️ There are still NaN values in the 'Gear Box' column for truly unidentifiable strings.")
    else:
        print("\n🎉 All 'Gear Box' values were successfully converted to a numeric representation.")

    #Optionally, you might not want to drop the temporary column for inspection
    if 'Gear Box_numeric' in df_standardised.columns:
        df_standardised.drop(columns=['Gear Box_numeric'], inplace=True)


def clean_and_transform(df_standardised):
    """Perform all cleaning and transformation operations"""
    try:
        df_transformed = df_standardised.copy()

        # ===== PROCESS KM COLUMN =====
        if 'Kms Driven' in df_standardised.columns:
            df_standardised['Kms Driven'] = clean_numeric_column(df_standardised['Kms Driven'], 'Int64')
            print("✅ Processed 'Kms Driven' column - removed commas and converted to integer")

        # ===== STANDARD TRANSFORMATIONS =====
        transformations = [
            ('owner', r'(\d+)', 'Int64'),
            ('Registration Year', r'(\d+)', 'Int64'),
            ('Seats', r'(\d+)', 'Int64'),
            ('Engine Displacement', r'(\d+)', 'Int64'),
            ('Mileage', r'(\d+\.?\d*)', float) # Extract numerical mileage
        ]

        for col, pattern, action in transformations:
            if col in df_transformed.columns:
                try:
                    if callable(action):
                        if pattern:
                            match = df_transformed[col].str.extract(pattern)
                            df_transformed[col] = match[0].apply(action) if not match.empty else None
                        else:
                            df_transformed[col] = df_transformed[col].apply(action)
                    else:
                        df_transformed[col] = clean_numeric_column(df_transformed[col], pattern, action)
                except Exception as e:
                    print(f"⚠️ Warning processing column {col}: {e}")

        return df_transformed
    except Exception as e:
        print(f"❌ Error in transformation: {e}")
        return None




df_standardised = clean_and_transform(df_standardised)

# Assuming df_standardised is your DataFrame

year_columns_to_convert = ['Year of Manufacture', 'Registration Year', 'modelYear']

for col in year_columns_to_convert:
    if col in df_standardised.columns:
        try:
            df_standardised[col] = pd.to_datetime(df_standardised[col], format='%Y').dt.to_period('Y')
            print(f"Converted '{col}' to Period[Y]")
        except ValueError:
            print(f"Could not convert '{col}' to Period[Y], keeping original type.")

print("\nDataFrame dtypes after converting year columns:")
print(df_standardised.dtypes)

print("\nDataFrame with Period[Y] dtype:")
print(df_standardised[year_columns_to_convert].head())


# Create a copy to detect data types without modifying the original
df_standardised_dtypes = df_standardised.copy().convert_dtypes()

print("Original DataFrame dtypes:")
print(df_standardised.dtypes)
print("\nDetected dtypes (without saving):")
print(df_standardised_dtypes.dtypes)
print("\nAutomatically detected the datatypes (in the copy)")


if df_standardised_dtypes is not None:
    print("\n the datatypes after transformation")
    print(df_standardised_dtypes.dtypes)
    df_standardised_dtypes.head()

    # Save the df_Standardising data
    os.makedirs("output", exist_ok=True)
    output_path = "output/banglore_standardised.csv"
    df_standardised_dtypes.to_csv(output_path, index=False)
    print(f"\n✅ Cleaned dataset saved to: {output_path}")


✅ Processed 'Kms Driven' column - removed non-numeric characters and converted to integer
✅ Processed 'price' column - attempted to convert 'Lakh' and 'Crore' values to numeric INR
Number of NaN values in 'price' column after conversion: 0

🎉 All identified price values were successfully converted to numeric.
✅ Processed 'Gear Box' column - attempted to extract the number of speeds (version 4 - modified for ML)
Number of NaN values in 'Gear Box' column after extraction: 0

🎉 All 'Gear Box' values were successfully converted to a numeric representation.
✅ Processed 'Kms Driven' column - removed commas and converted to integer
⚠️ Warning processing column Mileage: Can only use .str accessor with string values!
Converted 'Year of Manufacture' to Period[Y]
Converted 'Registration Year' to Period[Y]
Converted 'modelYear' to Period[Y]

DataFrame dtypes after converting year columns:
price                        float64
bt                            object
Kms Driven                     int64

In [12]:
df_standardised_dtypes.head(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.1,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.0,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.1,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248
5,1900000,SUV,20000,1,2020,17.1,KA04,Diesel,2020,2020,Third Party insurance,6,Manual,5,bangalore,1956
6,345000,Hatchback,37772,1,2017,20.63,KA05,Petrol,2017,2017,Comprehensive,5,Manual,5,bangalore,1198
7,1200000,SUV,30000,1,2021,18.15,KA51,Petrol,2021,2021,Third Party insurance,7,Automatic,5,bangalore,998
8,960000,Sedan,37000,1,2018,20.28,KA03,Petrol,2018,2018,Comprehensive,4,Automatic,5,bangalore,1462
9,585000,Hatchback,11949,1,2017,23.84,KA03,Petrol,2018,2017,Comprehensive,5,Manual,5,bangalore,1199


In [13]:
nan_summary = pd.DataFrame({
    'Column': df_standardised_dtypes .columns,
    'Data Type': df_standardised_dtypes .dtypes,
    'Non-Null Count': df_standardised_dtypes.notna().sum(),
    'NaN Count': df_standardised_dtypes .isna().sum(),
    'NaN Percentage': (df_standardised_dtypes.isna().mean() * 100).round(2)
}).reset_index(drop=True)

# Display ALL rows without truncation
with pd.option_context('display.max_rows', None, 'display.width', 1000):
    print(nan_summary)

                 Column       Data Type  Non-Null Count  NaN Count  NaN Percentage
0                 price           Int64            1481          0             0.0
1                    bt  string[python]            1481          0             0.0
2            Kms Driven           Int64            1481          0             0.0
3                 owner           Int64            1481          0             0.0
4   Year of Manufacture   period[Y-DEC]            1481          0             0.0
5               Mileage         Float64            1481          0             0.0
6                   RTO  string[python]            1481          0             0.0
7             Fuel Type  string[python]            1481          0             0.0
8     Registration Year   period[Y-DEC]            1481          0             0.0
9             modelYear   period[Y-DEC]            1481          0             0.0
10   Insurance Validity  string[python]            1481          0             0.0
11  

In [ ]:
import pandas as pd
import os

# File path of the input CSV file
input_file_path = r'C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output\banglore_standardised.csv'

# Output directory
output_dir = r'C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output'
output_file_name = 'Banglore_data_format.csv'
output_file_path = os.path.join(output_dir, output_file_name)

try:
    # Read the CSV file into a Pandas DataFrame
    df = pd.read_csv(input_file_path)

    # Create a mapping dictionary for column name changes
    column_name_mapping = {
        'BODY_TYPE': 'BODY_TYPE',  # Keep as is but ensure consistent capitalization
        'Kms Driven': 'Kilometers_Driven',
        'Gear Box': 'NUMBER_OF_GEARS',
        'City': 'CITY_NAME',
        'modelYear': 'MODEL_YEAR'
    }

    # Function to format column names
    def format_column_name(col_name):
        if col_name in column_name_mapping:
            return column_name_mapping[col_name].upper()
        else:
            return col_name.upper().replace(' ', '_')

    # Apply the formatting function to all column names
    df.columns = [format_column_name(col) for col in df.columns]

    # Save the modified DataFrame to a new CSV file
    df.to_csv(output_file_path, index=False, encoding='utf-8')

    print(f"✅ Successfully formatted column names and saved to: {output_file_path}")

except FileNotFoundError:
    print(f"❌ Error: Input file not found at: {input_file_path}")
except Exception as e:
    print(f"❌ An error occurred: {e}")

***d)	Encoding Categorical Variables: Convert categorical features into numerical values using encoding techniques.***

i)	Use one-hot encoding for nominal categorical variables.

ii)	Use label encoding or ordinal encoding for ordinal categorical variables.


In [17]:
import pandas as pd


file_path=r"C:\Users\USER\Desktop\Car-Dheko_Used_Car_Price_Prediction\output\banglore_standardised.csv"
df_Banglore_1=pd.read_csv(file_path)

df_Banglore_1.head(10)

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement
0,400000,Hatchback,120000,3,2015,23.10,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998
1,811000,SUV,32706,2,2018,17.00,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199
3,462000,Sedan,17794,1,2014,19.10,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248
5,1900000,SUV,20000,1,2020,17.10,KA04,Diesel,2020,2020,Third Party insurance,6,Manual,5,bangalore,1956
6,345000,Hatchback,37772,1,2017,20.63,KA05,Petrol,2017,2017,Comprehensive,5,Manual,5,bangalore,1198
7,1200000,SUV,30000,1,2021,18.15,KA51,Petrol,2021,2021,Third Party insurance,7,Automatic,5,bangalore,998
8,960000,Sedan,37000,1,2018,20.28,KA03,Petrol,2018,2018,Comprehensive,4,Automatic,5,bangalore,1462
9,585000,Hatchback,11949,1,2017,23.84,KA03,Petrol,2018,2017,Comprehensive,5,Manual,5,bangalore,1199


In [19]:
#label Encoding
import sklearn
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df_Banglore_1['price_encoded'] = le.fit_transform(df_Banglore_1['price'])
df_Banglore_1

,price,bt,Kms Driven,owner,Year of Manufacture,Mileage,RTO,Fuel Type,Registration Year,modelYear,Insurance Validity,Gear Box,Transmission,Seats,City,Engine Displacement,price_encoded
0,400000,Hatchback,120000,3,2015,23.10,KA51,Petrol,2015,2015,Third Party insurance,5,Manual,5,bangalore,998,102
1,811000,SUV,32706,2,2018,17.00,KA05,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1497,368
2,585000,Hatchback,11949,1,2018,23.84,KA03,Petrol,2018,2018,Comprehensive,5,Manual,5,bangalore,1199,226
3,462000,Sedan,17794,1,2014,19.10,KA53,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1197,142
4,790000,SUV,60000,1,2015,23.65,KA04,Diesel,2015,2015,Third Party insurance,5,Manual,5,bangalore,1248,355
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1476,1649000,SUV,156039,2,2012,12.55,KA01,Diesel,2012,2012,Comprehensive,5,Manual,7,bangalore,2982,541
1477,330000,Sedan,56000,2,2008,15.00,KA02,Petrol,2008,2008,Third Party insurance,5,Manual,5,bangalore,1586,66
1478,425000,Hatchback,42000,2,2014,19.40,KA03,Petrol,2014,2014,Comprehensive,5,Manual,5,bangalore,1198,119
1479,750000,Hatchback,93003,1,2018,22.54,KA03,Diesel,2018,2018,Comprehensive,6,Manual,5,bangalore,1396,336


In [20]:
df_Banglore_1.price.unique()

array([  400000,   811000,   585000,   462000,   790000,  1900000,
         345000,  1200000,   960000,   690000,   682000,   825000,
         595000,  1350000,  5595000,   521000,  1005000,   775000,
        2200000,   582000,  1090000,   457000,  4900000,   550000,
         570000,   220000,  4145000,   861000,  1785000,   803000,
        2565000,   674000,   349000,  1050000,  4425000,   406000,
        4965000,  1100000,   594000,   710000,   692000,   650000,
        2090000,   411000,  3675000,   610000,  1675000,   530000,
        7990000,   250000,   715000,  3500000,  2250000,   625000,
        2695000,   428000,   695000,   930000,   455000,  1195000,
        2175000,   442000,   500000,  3395000,   537000,   750000,
        1790000,   468000,   420000,   920000,   533000,   850000,
         751000,   388000,   490000,   494000,   525000,  2890000,
         365000,  1895000,  5990000,  2075000,   394000,   725000,
         240000,   835000,  1725000,   330000,   990000,   755

In [24]:
df_Banglore_1.owner.unique()

array([3, 2, 1, 4, 5])

In [30]:
import pandas as pd

# Assuming df_Banglore_1 is your DataFrame
if isinstance(df_Banglore_1, pd.DataFrame):
    for column in df_Banglore_1.columns:
        unique_values = df_Banglore_1[column].unique()
        print(f"Unique values in column '{column}':")
        print(unique_values)
        print("-" * 30)
else:
    print("df_Banglore_1 is not a Pandas DataFrame.")

Unique values in column 'price':
[  400000   811000   585000   462000   790000  1900000   345000  1200000
   960000   690000   682000   825000   595000  1350000  5595000   521000
  1005000   775000  2200000   582000  1090000   457000  4900000   550000
   570000   220000  4145000   861000  1785000   803000  2565000   674000
   349000  1050000  4425000   406000  4965000  1100000   594000   710000
   692000   650000  2090000   411000  3675000   610000  1675000   530000
  7990000   250000   715000  3500000  2250000   625000  2695000   428000
   695000   930000   455000  1195000  2175000   442000   500000  3395000
   537000   750000  1790000   468000   420000   920000   533000   850000
   751000   388000   490000   494000   525000  2890000   365000  1895000
  5990000  2075000   394000   725000   240000   835000  1725000   330000
   990000   755000  1750000  2083000   271000  1280000   436000  1250000
  2225000   475000   600000  2450000  1625000  1650000   249000   630000
  2599000   385000

In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Assuming your DataFrame is named 'df'

# oe = OrdinalEncoder(categories=[
#     ['Third Party insurance', 'Comprehensive'],  # Example order for Insurance Validity
#     ['Manual', 'Automatic'],                    # Example order for Gear Box/Transmission (subjective)
#     ['First', 'Second', 'Third']               # Example order for Owner (if categorical)
# ])

oe = OrdinalEncoder(categories=[['Manual','Automatic','Petrol' 'Diesel' 'LPG' 'CNG' 'Electric','Third Party insurance' 'Comprehensive' 'Third Party' 'Zero Dep' '2' '1'
 'Not Available',]])
df_Banglore_1['Transmission_encoded'] = oe.fit_transform(df_Banglore_1[['Transmission']])

# Select the columns you want to encode ordinally
columns_to_encode_ordinal = ['Insurance Validity', 'Gear Box', 'owner','Fuel Type',] # Adjust based on your actual column names

# Fit and transform the selected columns
df_Banglore_1[columns_to_encode_ordinal] = oe.fit_transform(df[columns_to_encode_ordinal])

print(df_Banglore_1.head())
print(df_Banglore_1.info())

ValueError: Shape mismatch: if categories is an array, it has to be of shape (n_features,).